# PS S6E6 - 008 Blend add007 bias check

Experiment: `exp_20260603_008_blend_add007_bias_check`

005のraw baseline相関・blend確認に、006 color index と 007 bias search を追加するNotebookです。

重要:
- `.npy` はすべて `/kaggle/input/datasets/mizushimatoshihiko/npy-files` から読み込みます。
- 学習は行いません。
- 目的は、007を002 rawの置換にするか、002 rawと007を両方残すか、004/006/003を残す価値があるかの確認です。

In [1]:
import os
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression, RidgeClassifier

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260603_008_blend_add007_bias_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {
        "key": "002_lgb_raw",
        "exp_id": "exp_20260601_002_lgb_strict_raw_baseline",
        "family": "LightGBM",
        "role": "raw_main_reference",
        "oof": "oof_lgb_strict_raw_proba.npy",
        "pred": "pred_lgb_strict_raw_proba.npy",
        "cv": 0.9569063401002502,
        "public_lb": 0.95800,
    },
    {
        "key": "003_cat_raw",
        "exp_id": "exp_20260601_003_cat_strict_raw_baseline",
        "family": "CatBoost",
        "role": "weak_reference",
        "oof": "oof_cat_strict_raw_proba.npy",
        "pred": "pred_cat_strict_raw_proba.npy",
        "cv": 0.9524581895303758,
        "public_lb": 0.95351,
    },
    {
        "key": "004_xgb_raw",
        "exp_id": "exp_20260603_004_xgb_strict_raw_baseline",
        "family": "XGBoost",
        "role": "aux_reference",
        "oof": "oof_xgb_strict_raw_proba.npy",
        "pred": "pred_xgb_strict_raw_proba.npy",
        "cv": 0.9557836270283273,
        "public_lb": 0.95638,
    },
    {
        "key": "006_lgb_color_min",
        "exp_id": "exp_20260603_006_lgb_color_index_min",
        "family": "LightGBM",
        "role": "risky_cv_only_candidate",
        "oof": "oof_lgb_color_index_min_proba.npy",
        "pred": "pred_lgb_color_index_min_proba.npy",
        "cv": 0.9570463209115138,
        "public_lb": 0.95663,
    },
    {
        "key": "007_lgb002_bias",
        "exp_id": "exp_20260603_007_lgb002_bias_search",
        "family": "LightGBM",
        "role": "current_mainline_candidate",
        "oof": "oof_exp_20260603_007_lgb002_bias_search_proba.npy",
        "pred": "pred_exp_20260603_007_lgb002_bias_search_proba.npy",
        "cv": 0.9586098708578418,
        "public_lb": 0.95921,
    },
]

BLEND_SETS = {
    "A_005_raw_recheck": ["002_lgb_raw", "003_cat_raw", "004_xgb_raw"],
    "B_replace_002_with_007": ["007_lgb002_bias", "003_cat_raw", "004_xgb_raw"],
    "C_keep_002_004_007": ["002_lgb_raw", "004_xgb_raw", "007_lgb002_bias"],
    "D_all_002_003_004_006_007": ["002_lgb_raw", "003_cat_raw", "004_xgb_raw", "006_lgb_color_min", "007_lgb002_bias"],
    "E_007_004_only": ["007_lgb002_bias", "004_xgb_raw"],
    "F_007_002_only": ["007_lgb002_bias", "002_lgb_raw"],
    "G_007_006_only": ["007_lgb002_bias", "006_lgb_color_min"],
    "H_007_only": ["007_lgb002_bias"],
}

print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)

OUTDIR: /kaggle/working/exp_20260603_008_blend_add007_bias_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files


In [2]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def load_npy_from_dataset(filename):
    path = NPY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing npy file: {path}\n"
            f"Edit MODEL_SPECS if the filename differs."
        )
    return path

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-4), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats.append(p)
            mats.append(logit_transform(p))
        elif mode == "raw_rank_logit":
            mats.append(p)
            mats.append(rank_normalize_matrix(p))
            mats.append(logit_transform(p))
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01), max_iter=200):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    cur_score = balanced_accuracy_score(y_true, cur.argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]

    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score = cur_score
            best_w = w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w = cand_w / cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score = score
                        best_w = cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score = best_score
                w = best_w
                improved = True
                hist.append({"iter": len(hist), "score": float(cur_score), "weights": w.copy().tolist()})

    final = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    return w, final, float(cur_score), hist

def nm_optimize_weights(y_true, probas, maxiter=1000):
    n = len(probas)
    def unpack(theta):
        e = np.exp(theta - np.max(theta))
        return e / e.sum()
    def objective(theta):
        w = unpack(theta)
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return -balanced_accuracy_score(y_true, p.argmax(axis=1))

    res = minimize(objective, np.zeros(n), method="Nelder-Mead",
                   options={"maxiter": maxiter, "xatol": 1e-7, "fatol": 1e-10})
    w = unpack(res.x)
    p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    score = balanced_accuracy_score(y_true, p.argmax(axis=1))
    return w, p, float(score), res

In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)
if not NPY_DIR.exists():
    raise FileNotFoundError(NPY_DIR)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof = {}
pred = {}
resolved_specs = []

for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = load_npy_from_dataset(spec["oof"])
    pred_path = load_npy_from_dataset(spec["pred"])

    oof_arr = np.load(oof_path).astype(np.float32)
    pred_arr = np.load(pred_path).astype(np.float32)

    validate_proba(f"{key} oof", oof_arr, len(train), n_classes)
    validate_proba(f"{key} pred", pred_arr, len(test), n_classes)

    oof[key] = oof_arr
    pred[key] = pred_arr

    row = dict(spec)
    row["resolved_oof_path"] = str(oof_path)
    row["resolved_pred_path"] = str(pred_path)
    row["oof_shape"] = list(oof_arr.shape)
    row["pred_shape"] = list(pred_arr.shape)
    resolved_specs.append(row)

model_keys = [s["key"] for s in MODEL_SPECS]

print("train:", train.shape)
print("test :", test.shape)
print("class_names:", class_names)
display(pd.DataFrame(resolved_specs))

train: (577347, 12)
test : (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']


,key,exp_id,family,role,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,raw_main_reference,oof_lgb_strict_raw_proba.npy,pred_lgb_strict_raw_proba.npy,0.956906,0.95800,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,weak_reference,oof_cat_strict_raw_proba.npy,pred_cat_strict_raw_proba.npy,0.952458,0.95351,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,aux_reference,oof_xgb_strict_raw_proba.npy,pred_xgb_strict_raw_proba.npy,0.955784,0.95638,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
3,006_lgb_color_min,exp_20260603_006_lgb_color_index_min,LightGBM,risky_cv_only_candidate,oof_lgb_color_index_min_proba.npy,pred_lgb_color_index_min_proba.npy,0.957046,0.95663,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
4,007_lgb002_bias,exp_20260603_007_lgb002_bias_search,LightGBM,current_mainline_candidate,oof_exp_20260603_007_lgb002_bias_search_proba.npy,pred_exp_20260603_007_lgb002_bias_search_proba...,0.958610,0.95921,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
# ============================================================
# 3. Individual scores
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)

    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
4,007_lgb002_bias,exp_20260603_007_lgb002_bias_search,LightGBM,current_mainline_candidate,0.958610,0.95921,0.958610,0.0,0.976693,0.966699,0.932438
3,006_lgb_color_min,exp_20260603_006_lgb_color_index_min,LightGBM,risky_cv_only_candidate,0.957046,0.95663,0.957046,0.0,0.978145,0.964872,0.928122
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,raw_main_reference,0.956906,0.95800,0.956906,0.0,0.978431,0.965000,0.927288
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,aux_reference,0.955784,0.95638,0.955784,0.0,0.978582,0.963318,0.925451
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,weak_reference,0.952458,0.95351,0.952458,0.0,0.977432,0.963498,0.916445


In [5]:
# ============================================================
# 4. Pairwise OOF correlation / disagreement / error overlap
# ============================================================

pair_rows = []
pred_labels = {k: proba_to_pred(oof[k]) for k in model_keys}
wrong_flags = {k: pred_labels[k] != y for k in model_keys}

for a, b in combinations(model_keys, 2):
    pa, pb = oof[a], oof[b]
    ya, yb = pred_labels[a], pred_labels[b]
    wrong_a, wrong_b = wrong_flags[a], wrong_flags[b]
    both = wrong_a & wrong_b
    either = wrong_a | wrong_b

    row = {
        "model_a": a,
        "model_b": b,
        "pearson_flat_proba": corr_pearson(flatten_proba(pa), flatten_proba(pb)),
        "spearman_flat_proba": corr_spearman(flatten_proba(pa), flatten_proba(pb)),
        "argmax_agreement": float((ya == yb).mean()),
        "argmax_disagreement": float((ya != yb).mean()),
        "wrong_rate_a": float(wrong_a.mean()),
        "wrong_rate_b": float(wrong_b.mean()),
        "both_wrong_rate": float(both.mean()),
        "either_wrong_rate": float(either.mean()),
        "error_jaccard": float(both.sum() / max(either.sum(), 1)),
        "a_wrong_b_correct_rate": float((wrong_a & ~wrong_b).mean()),
        "a_correct_b_wrong_rate": float((~wrong_a & wrong_b).mean()),
    }
    for ci, cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"] = corr_pearson(pa[:, ci], pb[:, ci])
        row[f"spearman_proba_{cls}"] = corr_spearman(pa[:, ci], pb[:, ci])
    pair_rows.append(row)

pairwise_df = pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
6,003_cat_raw,007_lgb002_bias,0.996401,0.981468,0.989230,0.010770,0.034134,0.031676,0.027607,0.038202,0.722661,0.006526,0.004069,0.995818,0.982473,0.996796,0.939826,0.992080,0.930138
5,003_cat_raw,006_lgb_color_min,0.996427,0.980368,0.989350,0.010650,0.034134,0.031716,0.027687,0.038162,0.725503,0.006447,0.004029,0.995812,0.981501,0.996792,0.931691,0.991897,0.923943
0,002_lgb_raw,003_cat_raw,0.996519,0.981796,0.989538,0.010462,0.031622,0.034134,0.027735,0.038020,0.729488,0.003887,0.006398,0.995920,0.982472,0.996843,0.939371,0.992158,0.928582
7,004_xgb_raw,006_lgb_color_min,0.997113,0.984696,0.990695,0.009305,0.032128,0.031716,0.027368,0.036475,0.750321,0.004760,0.004347,0.996786,0.984501,0.997045,0.942101,0.993628,0.926341
4,003_cat_raw,004_xgb_raw,0.997419,0.981125,0.990425,0.009575,0.034134,0.032128,0.028434,0.037828,0.751648,0.005700,0.003694,0.996561,0.982859,0.998087,0.946916,0.994371,0.904603
8,004_xgb_raw,007_lgb002_bias,0.997839,0.989643,0.992239,0.007761,0.032128,0.031676,0.028101,0.035703,0.787076,0.004027,0.003575,0.997826,0.990356,0.997545,0.965737,0.995291,0.940058
1,002_lgb_raw,004_xgb_raw,0.997944,0.989543,0.992682,0.007318,0.031622,0.032128,0.028298,0.035452,0.798222,0.003324,0.003830,0.997946,0.990355,0.997615,0.964777,0.995412,0.943056
9,006_lgb_color_min,007_lgb002_bias,0.998172,0.993331,0.992232,0.007768,0.031716,0.031676,0.027879,0.035512,0.785056,0.003837,0.003797,0.997533,0.990983,0.998744,0.964065,0.996050,0.973427
2,002_lgb_raw,006_lgb_color_min,0.998256,0.993739,0.992457,0.007543,0.031622,0.031716,0.027966,0.035372,0.790618,0.003656,0.003750,0.997628,0.990983,0.998798,0.964695,0.996149,0.973885
3,002_lgb_raw,007_lgb002_bias,0.999904,0.999807,0.997759,0.002241,0.031622,0.031676,0.030540,0.032758,0.932269,0.001083,0.001136,0.999877,1.000000,0.999941,0.999771,0.999854,0.999660


In [6]:
# ============================================================
# 5. Correlation matrices
# ============================================================

pearson_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
spearman_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
argmax_agreement = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
error_jaccard = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)

for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a, b] = corr_pearson(flatten_proba(oof[a]), flatten_proba(oof[b]))
        spearman_flat.loc[a, b] = corr_spearman(flatten_proba(oof[a]), flatten_proba(oof[b]))
        argmax_agreement.loc[a, b] = float((pred_labels[a] == pred_labels[b]).mean())
        both = wrong_flags[a] & wrong_flags[b]
        either = wrong_flags[a] | wrong_flags[b]
        error_jaccard.loc[a, b] = float(both.sum() / max(either.sum(), 1))

display(pearson_flat)
display(spearman_flat)
display(argmax_agreement)
display(error_jaccard)

pearson_flat.to_csv(OUTDIR / "corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR / "corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR / "matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR / "matrix_error_jaccard.csv")

,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias
002_lgb_raw,1.000000,0.996519,0.997944,0.998256,0.999904
003_cat_raw,0.996519,1.000000,0.997419,0.996427,0.996401
004_xgb_raw,0.997944,0.997419,1.000000,0.997113,0.997839
006_lgb_color_min,0.998256,0.996427,0.997113,1.000000,0.998172
007_lgb002_bias,0.999904,0.996401,0.997839,0.998172,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias
002_lgb_raw,1.000000,0.981796,0.989543,0.993739,0.999807
003_cat_raw,0.981796,1.000000,0.981125,0.980368,0.981468
004_xgb_raw,0.989543,0.981125,1.000000,0.984696,0.989643
006_lgb_color_min,0.993739,0.980368,0.984696,1.000000,0.993331
007_lgb002_bias,0.999807,0.981468,0.989643,0.993331,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias
002_lgb_raw,1.000000,0.989538,0.992682,0.992457,0.997759
003_cat_raw,0.989538,1.000000,0.990425,0.989350,0.989230
004_xgb_raw,0.992682,0.990425,1.000000,0.990695,0.992239
006_lgb_color_min,0.992457,0.989350,0.990695,1.000000,0.992232
007_lgb002_bias,0.997759,0.989230,0.992239,0.992232,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias
002_lgb_raw,1.000000,0.729488,0.798222,0.790618,0.932269
003_cat_raw,0.729488,1.000000,0.751648,0.725503,0.722661
004_xgb_raw,0.798222,0.751648,1.000000,0.750321,0.787076
006_lgb_color_min,0.790618,0.725503,0.750321,1.000000,0.785056
007_lgb002_bias,0.932269,0.722661,0.787076,0.785056,1.000000


In [7]:
# ============================================================
# 6. Blend diagnostics
# ============================================================

blend_rows = []
blend_pred_outputs = {}

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows.append(row)
    if test_p is not None:
        blend_pred_outputs[(set_name, method)] = test_p

for set_name, keys in BLEND_SETS.items():
    print(f"===== {set_name}: {keys} =====")

    probas = [oof[k] for k in keys]
    pred_probas = [pred[k] for k in keys]

    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred, weights=np.ones(len(keys)) / len(keys))

    if len(keys) >= 2:
        hc_w, hc_oof, hc_score, hc_hist = hill_climb_weights(y, probas)
        hc_pred = normalize_rows(sum(hc_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=hc_w, extra={"hc_history_len": len(hc_hist)})

        nm_w, nm_oof, nm_score, nm_res = nm_optimize_weights(y, probas)
        nm_pred = normalize_rows(sum(nm_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=nm_w, extra={"nm_success": bool(nm_res.success), "nm_message": str(nm_res.message)})

    for mode in ["raw", "raw_logit", "raw_rank_logit", "logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)

        try:
            lr = LogisticRegression(C=0.05, penalty="l2", solver="lbfgs", max_iter=2000, random_state=SEED, n_jobs=-1)
            lr.fit(X_meta, y)
            lr_oof = lr.predict_proba(X_meta)
            lr_pred = lr.predict_proba(X_test_meta)
            record_blend(set_name, f"logreg_{mode}", keys, lr_oof, lr_pred, extra={"C": 0.05})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")

        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            rc_oof = softmax_np(rc.decision_function(X_meta), axis=1)
            rc_pred = softmax_np(rc.decision_function(X_test_meta), axis=1)
            record_blend(set_name, f"ridgeclf_{mode}", keys, rc_oof, rc_pred, extra={"alpha": 10.0})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(blend_rows).sort_values("balanced_accuracy", ascending=False)
display(blend_df.head(80))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

===== A_005_raw_recheck: ['002_lgb_raw', '003_cat_raw', '004_xgb_raw'] =====
[WARN] logreg failed: A_005_raw_recheck raw_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_005_raw_recheck raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: A_005_raw_recheck raw_rank_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_005_raw_recheck raw_rank_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] logreg failed: A_005_raw_recheck logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_005_raw_recheck logit: Input X contains infinity or a value too large for dtype('float32').
===== B_replace_002_with_007: ['007_lgb002_bias', '003_cat_raw', '004_xgb_raw'] =====
[WARN] logreg failed: B_replace_002_with_007 raw_logit: Input X contains infinity or a value too large for dt

,set_name,method,keys,n_models,balanced_accuracy,weights_json,recall_GALAXY,recall_QSO,recall_STAR,hc_history_len,nm_success,nm_message,C,alpha
6,B_replace_002_with_007,hc_nonnegative_raw,"007_lgb002_bias,003_cat_raw,004_xgb_raw",3,0.958611,"{""007_lgb002_bias"": 0.9803921568627451, ""003_c...",0.976740,0.966631,0.932462,13.0,NaN,NaN,NaN,NaN
26,F_007_002_only,hc_nonnegative_raw,"007_lgb002_bias,002_lgb_raw",2,0.958610,"{""007_lgb002_bias"": 1.0, ""002_lgb_raw"": 0.0}",0.976693,0.966699,0.932438,8.0,NaN,NaN,NaN,NaN
47,H_007_only,avg,007_lgb002_bias,1,0.958610,"{""007_lgb002_bias"": 1.0}",0.976693,0.966699,0.932438,NaN,NaN,NaN,NaN,NaN
37,G_007_006_only,hc_nonnegative_raw,"007_lgb002_bias,006_lgb_color_min",2,0.958504,"{""007_lgb002_bias"": 0.7634928269848904, ""006_l...",0.977265,0.966426,0.931821,6.0,NaN,NaN,NaN,NaN
49,H_007_only,ridgeclf_raw,007_lgb002_bias,1,0.958476,None,0.976846,0.966554,0.932027,NaN,NaN,NaN,NaN,10.0
16,D_all_002_003_004_006_007,hc_nonnegative_raw,"002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_co...",5,0.958336,"{""002_lgb_raw"": 0.0, ""003_cat_raw"": 0.00403413...",0.977909,0.965956,0.931145,13.0,NaN,NaN,NaN,NaN
51,H_007_only,ridgeclf_raw_logit,007_lgb002_bias,1,0.958335,None,0.976942,0.966520,0.931543,NaN,NaN,NaN,NaN,10.0
9,B_replace_002_with_007,ridgeclf_raw,"007_lgb002_bias,003_cat_raw,004_xgb_raw",3,0.958324,None,0.977718,0.965879,0.931374,NaN,NaN,NaN,NaN,10.0
40,G_007_006_only,ridgeclf_raw,"007_lgb002_bias,006_lgb_color_min",2,0.958309,None,0.977699,0.965973,0.931253,NaN,NaN,NaN,NaN,10.0
53,H_007_only,ridgeclf_raw_rank_logit,007_lgb002_bias,1,0.958308,None,0.976952,0.966537,0.931435,NaN,NaN,NaN,NaN,10.0


In [8]:
# ============================================================
# 7. Save top blend submissions
# ============================================================

save_methods_priority = {"avg", "hc_nonnegative_raw", "nm_softmax_raw"}
top_save = blend_df[blend_df["method"].isin(save_methods_priority)].head(10).copy()

submission_rows = []

for _, row in top_save.iterrows():
    set_name = row["set_name"]
    method = row["method"]
    test_p = blend_pred_outputs.get((set_name, method))
    if test_p is None:
        continue

    labels = le.inverse_transform(test_p.argmax(axis=1))
    sub = pd.DataFrame({
        ID_COL: test[ID_COL].values,
        TARGET_COL: labels,
    })
    assert sub.shape[0] == sample.shape[0]
    assert sub.columns.tolist() == sample.columns.tolist()
    assert sub[ID_COL].equals(sample[ID_COL])

    fname = f"submission_exp_20260603_008_blend_add007_bias_check_{set_name}_{method}.csv".replace("/", "_")
    sub_path = OUTDIR / fname
    sub.to_csv(sub_path, index=False)

    pred_fname = f"pred_exp_20260603_008_blend_add007_bias_check_{set_name}_{method}_proba.npy".replace("/", "_")
    pred_path = OUTDIR / pred_fname
    np.save(pred_path, test_p.astype(np.float32))

    submission_rows.append({
        "set_name": set_name,
        "method": method,
        "balanced_accuracy": row["balanced_accuracy"],
        "submission": fname,
        "pred_proba": pred_fname,
    })

submission_df = pd.DataFrame(submission_rows)
display(submission_df)
submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)

,set_name,method,balanced_accuracy,submission,pred_proba
0,B_replace_002_with_007,hc_nonnegative_raw,0.958611,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
1,F_007_002_only,hc_nonnegative_raw,0.958610,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
2,H_007_only,avg,0.958610,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
3,G_007_006_only,hc_nonnegative_raw,0.958504,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
4,D_all_002_003_004_006_007,hc_nonnegative_raw,0.958336,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
5,G_007_006_only,nm_softmax_raw,0.958252,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
6,G_007_006_only,avg,0.958252,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
7,C_keep_002_004_007,hc_nonnegative_raw,0.958192,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
8,E_007_004_only,hc_nonnegative_raw,0.958191,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...
9,E_007_004_only,avg,0.957937,submission_exp_20260603_008_blend_add007_bias_...,pred_exp_20260603_008_blend_add007_bias_check_...


In [9]:
# ============================================================
# 8. Role judgement
# ============================================================

def get_ind_score(key):
    return float(individual_df.loc[individual_df["key"] == key, "recomputed_oof_cv"].iloc[0])

judgement = {
    "current_single_best": {
        "key": "007_lgb002_bias",
        "cv": get_ind_score("007_lgb002_bias"),
        "public_lb": 0.95921,
        "status": "current_mainline_single",
    },
    "main_questions": {
        "replace_002_with_007": "Check whether B_replace_002_with_007 beats A_005_raw_recheck and 007 single.",
        "keep_002_and_007_together": "Check whether C_keep_002_004_007 or F_007_002_only improves over 007 single.",
        "keep_004": "004 is the most plausible auxiliary model after 007.",
        "keep_006": "006 is risky_cv_only unless it improves a top blend.",
        "keep_003": "003 is weak; keep only if it improves a top blend.",
    },
    "initial_policy": {
        "007_lgb002_bias": "mainline_candidate",
        "002_lgb_raw": "raw_reference_or_possible_pair_with_007",
        "004_xgb_raw": "auxiliary_candidate",
        "006_lgb_color_min": "hold_risky_cv_only",
        "003_cat_raw": "weak_reference",
    },
}

save_json(judgement, OUTDIR / "role_judgement.json")
print(json.dumps(judgement, indent=2, ensure_ascii=False))

{
  "current_single_best": {
    "key": "007_lgb002_bias",
    "cv": 0.9586098708578418,
    "public_lb": 0.95921,
    "status": "current_mainline_single"
  },
  "main_questions": {
    "replace_002_with_007": "Check whether B_replace_002_with_007 beats A_005_raw_recheck and 007 single.",
    "keep_002_and_007_together": "Check whether C_keep_002_004_007 or F_007_002_only improves over 007 single.",
    "keep_004": "004 is the most plausible auxiliary model after 007.",
    "keep_006": "006 is risky_cv_only unless it improves a top blend.",
    "keep_003": "003 is weak; keep only if it improves a top blend."
  },
  "initial_policy": {
    "007_lgb002_bias": "mainline_candidate",
    "002_lgb_raw": "raw_reference_or_possible_pair_with_007",
    "004_xgb_raw": "auxiliary_candidate",
    "006_lgb_color_min": "hold_risky_cv_only",
    "003_cat_raw": "weak_reference"
  }
}


In [10]:
# ============================================================
# 9. Save summary / memo
# ============================================================

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Add 007 bias model to 005 raw/candidate blend and OOF correlation check",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "saved_submission_candidates": submission_df.to_dict(orient="records") if "submission_df" in globals() else [],
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add007_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 007 LGB002 bias search",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-03",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from the fixed npy-files Kaggle dataset folder, "
            "add 007 bias-search output to the 005 raw-baseline correlation/blend table, "
            "and check whether 007 should replace 002 raw or coexist with it."
        ),
        "success_criteria": [
            "load 002/003/004/006/007 OOF and pred npy files from /kaggle/input/datasets/mizushimatoshihiko/npy-files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations",
            "evaluate predefined blend sets",
            "save top blend submission candidates",
            "save memo and summary",
        ],
    },
    "inputs": {
        "npy_dir": str(NPY_DIR),
        "models": resolved_specs,
        "blend_sets": BLEND_SETS,
    },
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "corr_matrix_pearson_flat_proba": "corr_matrix_pearson_flat_proba.csv",
        "corr_matrix_spearman_flat_proba": "corr_matrix_spearman_flat_proba.csv",
        "matrix_argmax_agreement": "matrix_argmax_agreement.csv",
        "matrix_error_jaccard": "matrix_error_jaccard.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "saved_submission_candidates": "saved_submission_candidates.csv",
        "role_judgement": "role_judgement.json",
        "summary": "blend_add007_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "007 is the current mainline single/postprocess candidate. "
            "008 should decide whether 002 raw is replaced by 007, whether both remain, "
            "and whether 004 or 006 adds value after 007."
        ),
        "next_action": [
            "Review blend_diagnostics.csv",
            "Review 002 raw vs 007 biased correlation",
            "If a blend candidate beats 007 single CV by a meaningful margin, submit one candidate",
            "If not, keep 007 as mainline and move to next single-model bias or raw-blend bias experiment",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved files:
 - blend_add007_summary.json
 - blend_diagnostics.csv
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - pairwise_oof_correlation.csv
 - pred_exp_20260603_008_blend_add007_bias_check_B_replace_002_with_007_hc_nonnegative_raw_proba.npy
 - pred_exp_20260603_008_blend_add007_bias_check_C_keep_002_004_007_hc_nonnegative_raw_proba.npy
 - pred_exp_20260603_008_blend_add007_bias_check_D_all_002_003_004_006_007_hc_nonnegative_raw_proba.npy
 - pred_exp_20260603_008_blend_add007_bias_check_E_007_004_only_avg_proba.npy
 - pred_exp_20260603_008_blend_add007_bias_check_E_007_004_only_hc_nonnegative_raw_proba.npy
 - pred_exp_20260603_008_blend_add007_bias_check_F_007_002_only_hc_nonnegative_raw_proba.npy
 - pred_exp_20260603_008_blend_add007_bias_check_G_007_006_only_avg_proba.npy
 - pred_exp_20260603_008_blend_add007_bias_check_G_007_006_only_hc_nonnegative_raw_p